# Gemma 4 Model Capabilities

This notebook demonstrates how to use the Gemma 4 2B-IT multimodal model for various tasks, including text generation, image-to-text conversion, speech-to-text transcription, and automated speech translation.

## Setup and Installation

First, we need to install the necessary libraries and CUDA toolkit. The `transformers` library from Hugging Face is used to interact with the Gemma model, and `accelerate` along with `bitsandbytes` are used for model optimization, especially for efficient GPU usage.

In [ ]:
# Install the CUDA toolkit
!apt-get -y install cuda-toolkit-12-3

# Install or upgrade the Transformers library
!pip install --upgrade transformers

# Install accelerate and bitsandbytes for model optimization
!pip install accelerate bitsandbytes

## Model Initialization and Text Generation

This section initializes the Gemma 4 2B-IT model and its processor. A global `pipe` (pipeline) is created for `any-to-any` tasks, which allows it to handle both text and image inputs. The `text_generation` function then uses this pipeline to generate text based on a given prompt. Please note that the Gema 4 can have implementations of text generation such as auto-completing text, use of the dialog library, use prompt template, multu-turn conversation and  use system instructions

In [ ]:
from transformers import pipeline, GenerationConfig, AutoProcessor, AutoModelForMultimodalLM

MODEL_ID ="google/gemma-4-E2B-it"
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 512
gen_kwargs = dict(generation_config=config)

# For multimodal tasks, initialize model and processor once
# model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")
# processor = AutoProcessor.from_pretrained(MODEL_ID)

tasks = {
        "a2a": "any-to-any", # This task type likely handles both text and image+text for a multimodal model
        "text": "text-generation",
        "img_txt_2_txt": "image-text-to-text"
        }

# Define a helper function to create pipelines.
def loader_pipe(task: str) -> pipeline:
  pipe_instance = pipeline(
      task=task,
      model=MODEL_ID,
      device_map="auto",
      dtype="auto"
  )
  return pipe_instance

pipe = loader_pipe(tasks['a2a']) # Using 'any-to-any' to support both text and image+text inputs

def text_generation(prompt: str) -> str:
  # Use the globally initialized `pipe` for text generation
  messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
        ]
    }
  ]
  output = pipe(messages, return_full_text=False, generation_config=config)
  return output[0]['generated_text']

# def audio_to_text():

generated_text = text_generation("Autocomplete the following text. Roses are red...")

print(f"Generated Text: {generated_text}")


## Image-to-Text Generation

This function demonstrates how to extract descriptive text from an image by providing the image URL as input to the model. The model then generates a textual description of the image content. This is ofcourse a basic implementation but can also prompt with multiple images, OCR(model recognizes multilingual texts in an image), object detection etc

In [ ]:
# Generating text from an image
def text_from_image(image_link: str):
  messages = [
      {
          "role":"user",
          "content":[
              {"type": "image", "url": image_link},
              {"type": "text", "text": "What is shown in this image?"},
          ],
      },
      {
          "role":"assistant",
          "content": [
              {"type": "text", "text": "This image shows"},
          ],
      },
  ]
  output = pipe(text=messages, return_full_text=False, generate_kwargs=gen_kwargs)
  return output[0]['generated_text']

image_url = "https://ai.google.dev/static/gemma/docs/images/thali-indian-plate.jpg"

print(f"Generated Text from Image: {text_from_image(image_url)}")

## Audio-to-Text Transcription

This section showcases the model's ability to transcribe audio. It takes multiple audio files from a specified resource URL and provides a concise overview of their content in text format.

For audio data, there are a few things to note:-
- Audio is available on smaller models like E2B and E4B
- Gemma 4 takes audio in formats WAV and MP3
- Each second of an audio is 25 tokens
- Audio supports upto a maximum of 30 seconds(Python scripts to handle the splitting can come in handy)
- The audio data is processed as a single channel. So multi-channel may need to combine the sound data into a single channel
- For technical encoding:-
  - Sample Rate:- 16kHz using 32ms frames
  - Bit Depth:- 32-bit float format with normalized samples within the range of [-1, 1]


In [ ]:
# Speech to text

model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

RESOURCE_URL_PREFIX = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/sample-data/"

def speech_to_text(resource_url: str) -> str:
  messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Give me a concise overview of these audio files."},
            {"type": "text", "text": "journal1:"},
            {"type": "audio", "audio": f"{resource_url}journal1.wav"},
            {"type": "text", "text": "journal2:"},
            {"type": "audio", "audio": f"{resource_url}journal2.wav"},
            {"type": "text", "text": "journal3:"},
            {"type": "audio", "audio": f"{resource_url}journal3.wav"},
            {"type": "text", "text": "journal4:"},
            {"type": "audio", "audio": f"{resource_url}journal4.wav"},
            {"type": "text", "text": "journal5:"},
            {"type": "audio", "audio": f"{resource_url}journal5.wav"},
        ]
    }
  ]
  input_ids = processor.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True, return_dict=True,
      return_tensors="pt"
  )

  input_ids = input_ids.to(model.device, dtype=model.dtype)

  outputs = model.generate(**input_ids, max_new_tokens=1024)

  text = processor.batch_decode(
      outputs,
      skip_special_tokens=False,
      clean_up_tokenization_spaces=False
  )

  return text[0]

transcribed_text = speech_to_text(RESOURCE_URL_PREFIX)
print(f"Transcribed Text: {transcribed_text}")

## Automated Speech Translation

Here, the model is used for automated speech translation. It transcribes an audio segment into English and then translates it into Lusoga, demonstrating its multilingual capabilities.  The model also supports Automated Speech Translation and Automatic Speech Recognition

In [ ]:
# Automatic Speech Translation

def automated_speech_translation(audio_link: str)->str:
  messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Transcribe the following speech segment in English, then translate it into Lusoga. When formatting the answer, first output the transcription in English, then one newline, then output the string 'Lusoga: ', then the translation in Lusoga."},
            {"type": "audio", "audio": audio_link},
        ]
    }
  ]

  input_ids = processor.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True, return_dict=True,
      return_tensors="pt",
  )

  input_ids = input_ids.to(model.device, dtype=model.dtype)

  outputs = model.generate(**input_ids, max_new_tokens=64)

  text = processor.batch_decode(
      outputs,
      skip_special_tokens=False,
      clean_up_tokenization_spaces=False,
  )

  return text[0]

audio_url = "https://ai.google.dev/gemma/docs/audio/roses-are.wav"

translation_text = automated_speech_translation(audio_url)

print(f"Translated text: {translation_text}")


## Conclusion

This notebook has illustrated the versatility of the Gemma 4 2B-IT model in handling various multimodal tasks, including text generation, image-to-text conversion, speech-to-text transcription, and automated speech translation. Oh, it can describe videos too. The model's ability to process and generate content across different modalities makes it a powerful tool for diverse applications.